In [ ]:
import os
import pandas as pd
import torch
import librosa
import evaluate

from datasets import Dataset, DatasetDict
from dataclasses import dataclass

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
    TrainerCallback
)

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------

DATA_ROOT = "/share/documents/solomon/TigrignaASR"
MODEL_NAME = "agkphysics/wav2vec2-large-xlsr-53-amharic"

print("====================================")
print("Tigrinya XLSR Training Pipeline")
print("====================================")

# -------------------------------------------------------
# DEVICE
# -------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

# -------------------------------------------------------
# DISABLE NOTEBOOK PROGRESS CALLBACK
# -------------------------------------------------------

class DisableProgressBarCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        return control


# -------------------------------------------------------
# DATA LOADING
# -------------------------------------------------------

def load_split(split):

    print(f"\n[INFO] Loading {split} dataset...")

    text_path = f"{DATA_ROOT}/text/{split}/text"
    wavscp_path = f"{DATA_ROOT}/text/{split}/wav.scp"

    text_dict = {}

    with open(text_path) as f:
        for line in f:
            utt, sentence = line.strip().split(" ", 1)
            text_dict[utt] = sentence.lower().strip()

    rows = []

    with open(wavscp_path) as f:
        for line in f:
            utt, wav = line.strip().split(" ", 1)

            wav = wav.replace(
                "/share/documents/solomon/TigrignaASR",
                DATA_ROOT
            )

            if utt in text_dict and os.path.exists(wav):

                rows.append({
                    "audio_path": wav,
                    "sentence": text_dict[utt]
                })

    print(f"[INFO] {split} samples loaded: {len(rows)}")

    return Dataset.from_pandas(pd.DataFrame(rows))


print("\n[INFO] Preparing datasets...")

train_ds = load_split("train")
dev_ds = load_split("dev")
eval_ds = load_split("eval")

dataset = DatasetDict({
    "train": train_ds,
    "dev": dev_ds,
    "eval": eval_ds
})

print("\n[INFO] Dataset summary")
print(dataset)

# -------------------------------------------------------
# MODEL
# -------------------------------------------------------

print("\n[INFO] Loading XLSR model...")

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME)

model.config.ctc_zero_infinity = True
model.gradient_checkpointing_enable()

model.to(device)

print("[INFO] Model ready.")

# -------------------------------------------------------
# DATA COLLATOR
# -------------------------------------------------------

@dataclass
class DataCollatorCTC:

    processor: any

    def __call__(self, features):

        input_values = []
        labels = []

        for f in features:

            try:

                audio, sr = librosa.load(
                    f["audio_path"],
                    sr=16000
                )

                audio = audio[:16000 * 30]

                if len(audio) == 0:
                    continue

                inputs = self.processor(
                    audio,
                    sampling_rate=16000
                )

                input_values.append(
                    inputs.input_values[0]
                )

                labels.append(
                    self.processor.tokenizer(
                        f["sentence"]
                    ).input_ids
                )

            except Exception:
                print(f"[WARNING] Skipping bad audio: {f['audio_path']}")

        batch = self.processor.pad(
            {"input_values": input_values},
            return_tensors="pt"
        )

        label_batch = self.processor.tokenizer.pad(
            {"input_ids": labels},
            return_tensors="pt"
        )

        labels = label_batch["input_ids"]

        labels[
            labels == self.processor.tokenizer.pad_token_id
        ] = -100

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTC(processor)

print("[INFO] Data collator ready.")

# -------------------------------------------------------
# METRICS
# -------------------------------------------------------

print("\n[INFO] Loading WER metric...")

wer_metric = evaluate.load("wer")


def compute_metrics(pred):

    pred_logits = pred.predictions

    pred_ids = torch.argmax(
        torch.tensor(pred_logits),
        dim=-1
    )

    label_ids = pred.label_ids

    label_ids[
        label_ids == -100
    ] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids)

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {"wer": wer}


# -------------------------------------------------------
# TRAINING CONFIG
# -------------------------------------------------------

training_args = TrainingArguments(

    output_dir="./tigrinya_xlsr",

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=1e-5,

    logging_steps=50,

    eval_steps=500,

    save_steps=500,

    eval_strategy="steps",

    eval_accumulation_steps=2,

    fp16=torch.cuda.is_available(),

    save_total_limit=2,

    report_to="none",

    remove_unused_columns=False
)

print("[INFO] Training configuration ready.")

# -------------------------------------------------------
# TRAINER
# -------------------------------------------------------

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=dataset["train"],

    eval_dataset=dataset["dev"],

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[DisableProgressBarCallback()]
)

print("[INFO] Trainer initialized.")

# -------------------------------------------------------
# TRAINING
# -------------------------------------------------------

print("\n====================================")
print("[INFO] Starting training...")
print("====================================")

trainer.train()

print("\n====================================")
print("[INFO] Training finished.")
print("====================================")

# -------------------------------------------------------
# SAVE MODEL SAFELY (NO FREEZE)
# -------------------------------------------------------

print("\n[INFO] Saving trained model...")

trainer.save_model("./tigrinya_xlsr_final")
processor.save_pretrained("./tigrinya_xlsr_final")

print("[INFO] Model saved successfully.")

# -------------------------------------------------------
# CLEAN MEMORY
# -------------------------------------------------------

del trainer
del model

torch.cuda.empty_cache()

print("\n[INFO] Training completed successfully.")

/home/elshaday/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tigrinya XLSR Training Pipeline
[INFO] Using device: cuda

[INFO] Preparing datasets...

[INFO] Loading train dataset...
[INFO] train samples loaded: 11305

[INFO] Loading dev dataset...
[INFO] dev samples loaded: 511

[INFO] Loading eval dataset...
[INFO] eval samples loaded: 507

[INFO] Dataset summary
DatasetDict({
    train: Dataset({
        features: ['audio_path', 'sentence'],
        num_rows: 11305
    })
    dev: Dataset({
        features: ['audio_path', 'sentence'],
        num_rows: 511
    })
    eval: Dataset({
        features: ['audio_path', 'sentence'],
        num_rows: 507
    })
})

[INFO] Loading XLSR model...


Loading weights: 100%|█████████████████████| 424/424 [00:00<00:00, 16125.79it/s]


[INFO] Model ready.
[INFO] Data collator ready.

[INFO] Loading WER metric...
[INFO] Training configuration ready.
[INFO] Trainer initialized.

[INFO] Starting training...


Step,Training Loss,Validation Loss,Wer
500,10.706499,0.955409,0.710638
1000,9.471841,0.824123,0.652537
1500,8.512206,0.776982,0.628969
2000,8.451760,0.759063,0.621440


Writing model shards: 100%|███████████████████████| 1/1 [00:01<00:00,  1.09s/it]



[INFO] Training finished.

[INFO] Saving trained model...


Writing model shards: 100%|███████████████████████| 1/1 [00:01<00:00,  1.01s/it]

[INFO] Model saved successfully.

[INFO] Training completed successfully.


In [ ]:
import os
import torch
import librosa
import pandas as pd
import evaluate

from datasets import Dataset, DatasetDict
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------

DATA_ROOT = "/share/documents/solomon/TigrignaASR"
MODEL_PATH = "./tigrinya_xlsr_final"

# -------------------------------------------------------
# LOAD DATASET
# -------------------------------------------------------

def load_split(split):

    text_path = f"{DATA_ROOT}/text/{split}/text"
    wavscp_path = f"{DATA_ROOT}/text/{split}/wav.scp"

    text_dict = {}

    with open(text_path) as f:
        for line in f:
            utt, sentence = line.strip().split(" ", 1)
            text_dict[utt] = sentence.lower().strip()

    rows = []

    with open(wavscp_path) as f:
        for line in f:
            utt, wav = line.strip().split(" ", 1)

            wav = wav.replace(
                "/share/documents/solomon/TigrignaASR",
                DATA_ROOT
            )

            if utt in text_dict and os.path.exists(wav):

                rows.append({
                    "audio_path": wav,
                    "sentence": text_dict[utt]
                })

    return Dataset.from_pandas(pd.DataFrame(rows))


print("[INFO] Loading eval dataset...")

eval_ds = load_split("eval")

dataset = DatasetDict({
    "eval": eval_ds
})

print("[INFO] Eval samples:", len(dataset["eval"]))

# -------------------------------------------------------
# LOAD MODEL
# -------------------------------------------------------

print("[INFO] Loading trained model...")

processor = Wav2Vec2Processor.from_pretrained(MODEL_PATH)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

wer_metric = evaluate.load("wer")

# -------------------------------------------------------
# SAFE EVALUATION
# -------------------------------------------------------

print("\n[INFO] Evaluating on eval dataset...")

for sample in dataset["eval"]:

    try:

        audio, sr = librosa.load(sample["audio_path"], sr=16000)
        audio = audio[:16000 * 30]

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        )

        input_values = inputs.input_values.to(device)

        with torch.no_grad():
            logits = model(input_values).logits

        pred_ids = torch.argmax(logits, dim=-1)

        pred_str = processor.batch_decode(pred_ids)[0]

        wer_metric.add(
            prediction=pred_str,
            reference=sample["sentence"]
        )

    except Exception:
        print(f"[WARNING] Failed sample: {sample['audio_path']}")

wer = wer_metric.compute()

print("\n====================================")
print("Evaluation Results (Eval Set)")
print("====================================")

print("WER:", wer)

[INFO] Loading eval dataset...
[INFO] Eval samples: 507
[INFO] Loading trained model...


Loading weights: 100%|█████████████████████| 424/424 [00:00<00:00, 17350.77it/s]



[INFO] Evaluating on eval dataset...

Evaluation Results (Eval Set)
WER: 0.6619626926196269


In [ ]:
import torch
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import os

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------

MODEL_PATH = "./tigrinya_xlsr_final"
AUDIO_FILES = [
    "/share/documents/solomon/TigrignaASR/wav/test/001/TI001_1.wav",
    "/share/documents/solomon/TigrignaASR/wav/test/001/TI001_2.wav",
    "/share/documents/solomon/TigrignaASR/wav/test/001/TI001_3.wav"
]  # list your chosen files here

MAX_SECONDS = 30  # limit audio to 30 seconds

# -------------------------------------------------------
# LOAD MODEL
# -------------------------------------------------------

print("[INFO] Loading model...")
processor = Wav2Vec2Processor.from_pretrained(MODEL_PATH)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# -------------------------------------------------------
# TRANSCRIBE SELECTED AUDIO FILES
# -------------------------------------------------------

for audio_file in AUDIO_FILES:
    if not os.path.exists(audio_file):
        print(f"[WARNING] File not found: {audio_file}")
        continue

    print(f"\n[INFO] Processing {audio_file} ...")

    # Load audio
    audio, sr = librosa.load(audio_file, sr=16000)
    audio = audio[:16000 * MAX_SECONDS]  # truncate if needed

    # Prepare input
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    # Predict
    with torch.no_grad():
        logits = model(input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(pred_ids)[0]

    # Print result
    print("====================================")
    print("Transcription")
    print("====================================")
    print(transcription)

[INFO] Loading model...


Loading weights: 100%|██████████████████████| 424/424 [00:00<00:00, 3143.74it/s]



[INFO] Processing /share/documents/solomon/TigrignaASR/wav/test/001/TI001_1.wav ...
Transcription
ይ<unk>ዳ ተ<unk>ዘን ደጌታታ <unk>ልሊ ከከሊ ተኸዲኖም ኣብ ምድሪ ይዳኑ ኣለው ብ<unk>ያት እየርሳዮሩሳሌም ዋ<unk> ይብር ኣሎ

[INFO] Processing /share/documents/solomon/TigrignaASR/wav/test/001/TI001_2.wav ...
Transcription
ደቂ ብንያ በበኣሌቶም ኣብ ቤላ ኣሌት በላ<unk>ውያን

[INFO] Processing /share/documents/solomon/TigrignaASR/wav/test/001/TI001_3.wav ...
Transcription
ይ<unk>ን እምበር ኣብ ምድለው ም<unk>ራፍ ክልድተ ሽ<unk>ን ሸሞንተን <unk>መት ምህረ ዝተኸየደ መድረ<unk> ም<unk>ራይ ው<unk>ይ ቀርሲ ህዝቢ


In [ ]:
import os
import pandas as pd
import torch
import librosa
import evaluate

from datasets import Dataset, DatasetDict
from dataclasses import dataclass

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
    TrainerCallback
)

# ---------------- CONFIG ----------------
DATA_ROOT = "/share/documents/solomon/TigrignaASR"
MODEL_NAME = "agkphysics/wav2vec2-large-xlsr-53-amharic"

print("Tigrinya ASR Training Pipeline (GPU, 8GB VRAM)")

# ---------------- DEVICE ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[INFO] Device:", device)
if device.type == "cuda":
    print("[INFO] GPU:", torch.cuda.get_device_name(0))

# ---------------- CALLBACK ----------------
class DisableProgressBarCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        return control

# ---------------- DATA LOADER ----------------
def load_split(split):
    text_path = f"{DATA_ROOT}/text/{split}/text"
    wavscp_path = f"{DATA_ROOT}/text/{split}/wav.scp"

    text_dict = {}
    with open(text_path) as f:
        for line in f:
            utt, sentence = line.strip().split(" ", 1)
            text_dict[utt] = sentence.lower().strip()

    rows = []
    with open(wavscp_path) as f:
        for line in f:
            utt, wav = line.strip().split(" ", 1)
            wav = wav.replace("/share/documents/solomon/TigrignaASR", DATA_ROOT)
            if utt in text_dict and os.path.exists(wav):
                rows.append({"audio_path": wav, "sentence": text_dict[utt]})

    print(f"[INFO] {split} samples: {len(rows)}")
    return Dataset.from_pandas(pd.DataFrame(rows))

# Load datasets
train_ds = load_split("train")
dev_ds = load_split("dev")
eval_ds = load_split("eval")

dataset = DatasetDict({"train": train_ds, "dev": dev_ds, "eval": eval_ds})

# ---------------- MODEL ----------------
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME)
model.config.ctc_zero_infinity = True
model.freeze_feature_encoder()  # freeze encoder to save memory
model.to(device)

# ---------------- DATA COLLATOR ----------------
@dataclass
class DataCollatorCTC:
    processor: any

    def __call__(self, features):
        input_values, labels = [], []
        for f in features:
            try:
                audio, sr = librosa.load(f["audio_path"], sr=16000)
                audio = audio[:16000 * 20]  # limit to 20 sec
                if len(audio) == 0:
                    continue
                inputs = self.processor(audio, sampling_rate=16000)
                input_values.append(inputs.input_values[0])
                labels.append(self.processor.tokenizer(f["sentence"]).input_ids)
            except:
                print("[WARNING] bad audio:", f["audio_path"])

        batch = self.processor.pad({"input_values": input_values}, return_tensors="pt")
        label_batch = self.processor.tokenizer.pad({"input_ids": labels}, return_tensors="pt")
        labels = label_batch["input_ids"]
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTC(processor)

# ---------------- METRICS ----------------
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = torch.argmax(torch.tensor(pred_logits), dim=-1)
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}

# ---------------- TRAINING CONFIG ----------------
training_args = TrainingArguments(
    output_dir="./tigrinya_asr",
    per_device_train_batch_size=1,  # very small for 8GB GPU
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # simulate larger batch
    learning_rate=1e-5,
    logging_steps=50,
    eval_steps=500,
    save_steps=500,
    eval_strategy="steps",
    fp16=True,  # mixed precision
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["dev"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[DisableProgressBarCallback()]
)

# ---------------- TRAINING ----------------
print("[INFO] Trainer ready. Starting training...")
trainer.train()

# ---------------- SAVE MODEL ----------------
print("[INFO] Saving model...")
trainer.save_model("./tigrinya_asr_final")
processor.save_pretrained("./tigrinya_asr_final")

del trainer, model
torch.cuda.empty_cache()
print("[INFO] Training completed successfully.")

# ---------------- EVALUATION ----------------
# Load model again for evaluation
processor = Wav2Vec2Processor.from_pretrained("./tigrinya_asr_final")
model = Wav2Vec2ForCTC.from_pretrained("./tigrinya_asr_final")
model.to(device)
model.eval()



/home/elshaday/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tigrinya ASR Training Pipeline (GPU, 8GB VRAM)
[INFO] Device: cuda
[INFO] GPU: NVIDIA GeForce RTX 4060 Laptop GPU
[INFO] train samples: 11305
[INFO] dev samples: 511
[INFO] eval samples: 507


Loading weights: 100%|██████████████████████| 424/424 [00:00<00:00, 3733.96it/s]


[INFO] Trainer ready. Starting training...


Step,Training Loss,Validation Loss,Wer
500,11.078506,0.966504,0.720786
1000,9.831623,0.840836,0.659738
1500,8.781003,0.772962,0.626187
2000,8.453947,0.738526,0.602128
2500,8.215070,0.714152,0.590998
3000,7.963669,0.697608,0.581342
3500,8.036354,0.686171,0.574632
4000,7.660838,0.680629,0.568085


Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  1.00it/s]


In [ ]:
import torch
import librosa
import evaluate
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

MODEL_PATH = "./tigrinya_asr_final"
DATA_ROOT = "/share/documents/solomon/TigrignaASR"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading model for evaluation...")

processor = Wav2Vec2Processor.from_pretrained(MODEL_PATH)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_PATH).to(device)
model.eval()

wer_metric = evaluate.load("wer")


# ----------- LOAD DATA -----------
def load_split(split):
    text_path = f"{DATA_ROOT}/text/{split}/text"
    wavscp_path = f"{DATA_ROOT}/text/{split}/wav.scp"

    text_dict = {}
    with open(text_path) as f:
        for line in f:
            utt, sentence = line.strip().split(" ", 1)
            text_dict[utt] = sentence.lower().strip()

    rows = []
    with open(wavscp_path) as f:
        for line in f:
            utt, wav = line.strip().split(" ", 1)

            wav = wav.replace("/share/documents/solomon/TigrignaASR", DATA_ROOT)

            if utt in text_dict:
                rows.append((wav, text_dict[utt]))

    return rows


# ----------- EVALUATION FUNCTION -----------

def evaluate_dataset(data, name="eval"):

    predictions = []
    references = []

    print(f"Evaluating {name} set ({len(data)} samples)")

    for wav_path, ref in data:

        try:
            audio, sr = librosa.load(wav_path, sr=16000)

            # limit audio length (20 sec)
            audio = audio[:16000 * 20]

            inputs = processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt",
                padding=True
            )

            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                logits = model(**inputs).logits

            pred_ids = torch.argmax(logits, dim=-1)

            pred = processor.batch_decode(pred_ids)[0]

            predictions.append(pred.lower())
            references.append(ref.lower())

        except Exception as e:
            print("Skipping file:", wav_path)
            print(e)

    wer = wer_metric.compute(predictions=predictions, references=references)

    print(f"\n{name} WER: {wer:.4f}")

    return wer


# ----------- RUN EVALUATION -----------

eval_data = load_split("eval")
evaluate_dataset(eval_data, "eval")

# Optional test evaluation
try:
    test_data = load_split("test")
    evaluate_dataset(test_data, "test")
except:
    print("Test set not found")

/home/elshaday/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model for evaluation...


Loading weights: 100%|██████████████████████| 424/424 [00:00<00:00, 3484.95it/s]


Evaluating eval set (507 samples)

eval WER: 0.6273
Test set not found


In [ ]:
import os
import pandas as pd
import torch
import librosa
import evaluate
import numpy as np

from datasets import Dataset, DatasetDict
from dataclasses import dataclass

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)

# ---------------- CONFIG ----------------
DATA_ROOT = "/share/documents/solomon/TigrignaASR"
MODEL_NAME = "agkphysics/wav2vec2-large-xlsr-53-amharic"

MAX_AUDIO_LEN = 15  # seconds (reduces VRAM)

print("Optimized Tigrinya ASR Training")

# ---------------- DEVICE ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------------- DATA LOADER ----------------
def load_split(split):

    text_path = f"{DATA_ROOT}/text/{split}/text"
    wavscp_path = f"{DATA_ROOT}/text/{split}/wav.scp"

    text_dict = {}

    with open(text_path) as f:
        for line in f:
            utt, sentence = line.strip().split(" ",1)
            text_dict[utt] = sentence.lower().strip()

    rows = []

    with open(wavscp_path) as f:
        for line in f:
            utt, wav = line.strip().split(" ",1)

            wav = wav.replace("/share/documents/solomon/TigrignaASR", DATA_ROOT)

            if utt in text_dict and os.path.exists(wav):
                rows.append({
                    "audio_path": wav,
                    "sentence": text_dict[utt]
                })

    print(split,len(rows))

    return Dataset.from_pandas(pd.DataFrame(rows))


train_ds = load_split("train")
dev_ds = load_split("dev")
eval_ds = load_split("eval")

dataset = DatasetDict({
    "train": train_ds,
    "dev": dev_ds,
    "eval": eval_ds
})

# ---------------- MODEL ----------------
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id
)

# memory saving
model.gradient_checkpointing_enable()

# freeze only feature extractor
model.freeze_feature_encoder()

# enable specaugment
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05

model.to(device)

# ---------------- AUDIO LOADER ----------------
def load_audio(path):

    speech, sr = librosa.load(path, sr=16000)

    speech = speech[:16000*MAX_AUDIO_LEN]

    speech = speech / np.max(np.abs(speech))  # normalize

    return speech


# ---------------- DATA COLLATOR ----------------
@dataclass
class DataCollatorCTC:

    processor: any

    def __call__(self, features):

        input_values = []
        labels = []

        for f in features:

            try:

                audio = load_audio(f["audio_path"])

                inputs = self.processor(
                    audio,
                    sampling_rate=16000
                )

                input_values.append(inputs.input_values[0])

                labels.append(
                    self.processor.tokenizer(
                        f["sentence"]
                    ).input_ids
                )

            except:
                continue

        batch = self.processor.pad(
            {"input_values": input_values},
            padding=True,
            return_tensors="pt"
        )

        label_batch = self.processor.tokenizer.pad(
            {"input_ids": labels},
            padding=True,
            return_tensors="pt"
        )

        labels = label_batch["input_ids"]

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTC(processor)

# ---------------- METRIC ----------------
wer_metric = evaluate.load("wer")

def compute_metrics(pred):

    pred_ids = torch.argmax(
        torch.tensor(pred.predictions),
        dim=-1
    )

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids)

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {"wer": wer}


# ---------------- TRAINING ----------------
training_args = TrainingArguments(

    output_dir="./tigrinya_asr",

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=3e-5,

    warmup_steps=1000,

    num_train_epochs=20,

    eval_strategy="steps",

    save_steps=500,
    eval_steps=500,

    logging_steps=50,

    fp16=True,

    save_total_limit=2,

    report_to="none",

    remove_unused_columns=False
)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["dev"],

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print("Training starting...")

trainer.train()

print("Saving...")

trainer.save_model("./tigrinya_asr_final2")
processor.save_pretrained("./tigrinya_asr_final2")

/home/elshaday/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Optimized Tigrinya ASR Training
Device: cuda
train 11305
dev 511
eval 507


Loading weights: 100%|██████████████████████| 424/424 [00:00<00:00, 3030.33it/s]


Training starting...


Step,Training Loss,Validation Loss,Wer
500,10.979703,0.988604,0.728478
1000,8.645573,0.742635,0.603273
1500,7.134781,0.626580,0.531097
2000,6.691317,0.586543,0.492635
2500,6.453344,0.542682,0.470704
3000,5.995230,0.529018,0.451391
3500,6.140902,0.504111,0.435352
4000,5.730909,0.487772,0.422750
4500,5.385862,0.482152,0.411620
5000,5.228457,0.477746,0.411293


Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  1.14it/s]


Saving...


Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  1.24it/s]


['./tigrinya_asr_final2/processor_config.json']

In [ ]:
import os
import torch
import librosa
import numpy as np
import evaluate
from tqdm import tqdm

from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

# ---------------- CONFIG ----------------
DATA_ROOT = "/share/documents/solomon/TigrignaASR"
MODEL_PATH = "./tigrinya_asr_final2"

MAX_AUDIO_LEN = 10   # shorter audio = less VRAM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------------- LOAD MODEL ----------------
processor = Wav2Vec2Processor.from_pretrained(MODEL_PATH)

model = Wav2Vec2ForCTC.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

# ---------------- LOAD EVAL LIST ----------------
text_path = f"{DATA_ROOT}/text/eval/text"
wavscp_path = f"{DATA_ROOT}/text/eval/wav.scp"

text_dict = {}

with open(text_path) as f:
    for line in f:
        utt, sentence = line.strip().split(" ",1)
        text_dict[utt] = sentence.lower().strip()

eval_rows = []

with open(wavscp_path) as f:
    for line in f:
        utt, wav = line.strip().split(" ",1)

        wav = wav.replace(
            "/share/documents/solomon/TigrignaASR",
            DATA_ROOT
        )

        if utt in text_dict and os.path.exists(wav):
            eval_rows.append((wav, text_dict[utt]))

print("Eval samples:", len(eval_rows))

# ---------------- METRIC ----------------
wer_metric = evaluate.load("wer")

predictions = []
references = []

# ---------------- EVALUATION LOOP ----------------
for wav, text in tqdm(eval_rows):

    try:

        speech, sr = librosa.load(wav, sr=16000)

        speech = speech[:16000 * MAX_AUDIO_LEN]

        speech = speech / np.max(np.abs(speech))

        inputs = processor(
            speech,
            sampling_rate=16000,
            return_tensors="pt"
        )

        input_values = inputs.input_values.to(device)

        with torch.no_grad():

            logits = model(input_values).logits

        pred_ids = torch.argmax(logits, dim=-1)

        pred = processor.batch_decode(pred_ids)[0]

        predictions.append(pred)
        references.append(text)

        # free GPU memory immediately
        del input_values, logits, pred_ids
        torch.cuda.empty_cache()

    except:
        continue

# ---------------- FINAL WER ----------------
wer = wer_metric.compute(
    predictions=predictions,
    references=references
)

print("\nFINAL WER:", wer)

# ---------------- SAMPLE OUTPUT ----------------
print("\nSample Predictions\n")

for i in range(min(10, len(predictions))):

    print("REF :", references[i])
    print("PRED:", predictions[i])
    print("------")

/home/elshaday/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


Loading weights: 100%|██████████████████████| 424/424 [00:00<00:00, 2968.70it/s]


Eval samples: 507


100%|█████████████████████████████████████████| 507/507 [00:39<00:00, 12.79it/s]


FINAL WER: 0.51889699918897

Sample Predictions

REF : ዳዊት ከኣ ንጊብዖናውያን ንርስቲ እግዚኣብሄር ክትምርቑስ እንታይ ክገብረልኩም ብምንታይከ ዕርቂ ኽገብር እየ በሎም
PRED: ዳዊት ከኣ ንጊብ<unk> ናውያን ንርስቲ እግዚኣብሄር ክትምርቁስ እንታይ ክገብረልኩም ብምንታይኪ<unk>ርቂ ክገብር እየ በሎም
------
REF : ጅግንነት ወየንቲ ጅግንነት ሽግ ወየንቲ ጅግንነት ተጋደልቲ ጅግንነት ደቂ ኣንስትዮ ጀግንነት መናእሰይ ጀግንነት ጠቕላላ
PRED: ጅግንነት ወየንቲ ጅግንነት ሽግ ወየንቲ ጅግነት ተጋደልቲ ጅግንነት ደቂ ኣንስትዮ ጅግንነት መናእሰይ ጅግንነት ጠ<unk>ላላ
------
REF : ገምጋም ከምዝተኻተቱ ሓቢረን
PRED: ገምጋም ከም ዝተ<unk>ተቱ <unk>ቢረን
------
REF : ገምጋምን ምፅራይን ማእኸላይ ኣመራርሓ ብዝምልከት ቀንዲ መድረኽ ኮይኑ ዝተወሰደ በቢ ወረድኡ ዝገበሮ ቃልሲ ኮይኑ
PRED: ገምጋምን ም<unk>ራይን ማእኸላይ ኣመራር<unk> ብዝምልከት ቀንዲ መድረ<unk> ኮይኑ ዝተወሰደ ብብወረደኡ ዝገበሮ ቃልሲ ኮይኑ
------
REF : ደቂ ቀሀት ዓምራምን ዮፅሀርን ሔብሮንን ዑዚኤልን ኣርባዕተ
PRED: ደቂ ቅሀት <unk>ምራምን የ<unk>ሀርን <unk>ብሮንን <unk>ዚ ኤልን ኣርባ<unk>ተ
------
REF : ዶስ ኣዒንትና ብሰሪ እቲ ንመዋእል ዘሕለፍናዮ መስገደል ንብዓተን ፀንቂቐን እየን ኣይፋል
PRED: ዶስ ኣ<unk>ንትና ብሰሪ እቲ ንመዋእል ዘ<unk>ለፍናዮ መስገደል ንባ<unk>ትን <unk>ንቂ<unk>ን እየን ኣይፋልን
------
REF : ዳዊት ኪዛረብ ናብ ኬብሮን ከደ
PRED: ዳዊት ግዛረብ ናብ ኬብሮን ከደ
------
REF : ገብረመስቀል የረድእ
PRED: ገብረ መስቃል የረድእ
------


In [ ]:
import os
import pandas as pd
import torch
import librosa
import evaluate
import numpy as np

from datasets import Dataset, DatasetDict
from dataclasses import dataclass

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)

# ---------------- CONFIG ----------------
DATA_ROOT = "/share/documents/solomon/TigrignaASR"
MODEL_NAME = "agkphysics/wav2vec2-large-xlsr-53-amharic"

MAX_AUDIO_LEN = 12  # seconds (helps VRAM)

print("Optimized Tigrinya ASR Training (8GB VRAM Safe)")

# ---------------- DEVICE ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ---------------- TEXT NORMALIZATION ----------------
def normalize_text(text):

    text = text.lower()

    # remove punctuation that tokenizer may not support
    text = text.replace(",", "")
    text = text.replace(".", "")
    text = text.replace("።", "")
    text = text.replace("፣", "")
    text = text.replace("፥", "")
    text = text.replace("፦", "")

    # collapse spaces
    text = " ".join(text.split())

    return text


# ---------------- DATA LOADER ----------------
def load_split(split):

    text_path = f"{DATA_ROOT}/text/{split}/text"
    wavscp_path = f"{DATA_ROOT}/text/{split}/wav.scp"

    text_dict = {}

    with open(text_path, encoding="utf-8") as f:
        for line in f:

            utt, sentence = line.strip().split(" ", 1)

            sentence = normalize_text(sentence)

            text_dict[utt] = sentence

    rows = []

    with open(wavscp_path, encoding="utf-8") as f:
        for line in f:

            utt, wav = line.strip().split(" ", 1)

            wav = wav.replace(
                "/share/documents/solomon/TigrignaASR",
                DATA_ROOT
            )

            if utt in text_dict and os.path.exists(wav):

                rows.append({
                    "audio_path": wav,
                    "sentence": text_dict[utt]
                })

    print(split, len(rows))

    return Dataset.from_pandas(pd.DataFrame(rows))


train_ds = load_split("train")
dev_ds = load_split("dev")
eval_ds = load_split("eval")

dataset = DatasetDict({
    "train": train_ds,
    "dev": dev_ds,
    "eval": eval_ds
})


# ---------------- PROCESSOR ----------------
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)


# ---------------- MODEL ----------------
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id
)

# memory saving
model.gradient_checkpointing_enable()

# freeze feature encoder (recommended)
model.freeze_feature_encoder()

# SpecAugment
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05

model.to(device)


# ---------------- AUDIO LOADER ----------------
def load_audio(path):

    speech, sr = librosa.load(path, sr=16000)

    speech = speech[:16000 * MAX_AUDIO_LEN]

    speech = speech / np.max(np.abs(speech))

    return speech


# ---------------- DATA COLLATOR ----------------
@dataclass
class DataCollatorCTC:

    processor: any

    def __call__(self, features):

        input_values = []
        labels = []

        for f in features:

            try:

                audio = load_audio(f["audio_path"])

                inputs = self.processor(
                    audio,
                    sampling_rate=16000
                )

                input_values.append(inputs.input_values[0])

                labels.append(
                    self.processor.tokenizer(
                        f["sentence"]
                    ).input_ids
                )

            except:
                continue

        batch = self.processor.pad(
            {"input_values": input_values},
            padding=True,
            return_tensors="pt"
        )

        label_batch = self.processor.tokenizer.pad(
            {"input_ids": labels},
            padding=True,
            return_tensors="pt"
        )

        labels = label_batch["input_ids"]

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTC(processor)


# ---------------- METRIC ----------------
wer_metric = evaluate.load("wer")

def compute_metrics(pred):

    pred_ids = torch.argmax(
        torch.tensor(pred.predictions),
        dim=-1
    )

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(
        pred_ids,
        group_tokens=True
    )

    label_str = processor.batch_decode(
        label_ids,
        group_tokens=False
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {"wer": wer}


# ---------------- TRAINING (8GB SAFE) ----------------
training_args = TrainingArguments(

    output_dir="./tigrinya_asr",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=16,

    learning_rate=3e-5,

    warmup_steps=1000,

    num_train_epochs=20,

    eval_strategy="steps",

    save_steps=500,
    eval_steps=500,

    logging_steps=50,

    fp16=True,

    save_total_limit=2,

    report_to="none",

    remove_unused_columns=False
)


trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=dataset["train"],

    eval_dataset=dataset["dev"],

    data_collator=data_collator,

    compute_metrics=compute_metrics
)


print("Training starting...")

trainer.train()

print("Saving model...")

trainer.save_model("./tigrinya_asr_final")
processor.save_pretrained("./tigrinya_asr_final")

print("Training complete.")

/home/elshaday/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Optimized Tigrinya ASR Training (8GB VRAM Safe)
Device: cuda
train 11305
dev 511
eval 507


Loading weights: 100%|█████████████████████| 424/424 [00:00<00:00, 11940.76it/s]


Training starting...


Step,Training Loss,Validation Loss,Wer
500,22.410518,1.021536,0.726514
1000,17.653796,0.788504,0.598691
1500,14.993646,0.673319,0.524386
2000,13.848795,0.635327,0.479869
2500,13.471017,0.585150,0.458920
3000,12.499135,0.571852,0.445172
3500,12.614514,0.534400,0.422913
4000,11.799741,0.534488,0.416203
4500,11.170540,0.532490,0.404910
5000,10.693972,0.518635,0.400164


Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  1.23it/s]
